In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import os, sys
import torch
import numpy as np
import pandas as pd 
import random
import json
import re

from pathlib import Path

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

def init_cfg(folder):
    cfg = Config.fromfile(f'{folder}/vfnet_r18.py')
    return cfg

def extract_first_band(path):
    # Regular expression to extract the first number after the first 'b'
    match = re.search(r'perfect_b(\d+)', path)

    # Extracted value
    if match:
        first_b_number = match.group(1)
        return int(first_b_number)

def list_weights(folder):
    return [x for x in Path(folder).rglob('*.pth') if 'best' not in str(x)]

def get_weight_b(band_sel):
    weigths = list_weights(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band_sel}')
    assert len(weigths) > 0, 'No weights found'
    weight_path = weigths[0]
    return weight_path


def set_NoiseTesting_config(weight_path, sensor: str, SNR: float, mtf_at_fe: float, Seed: int = 18):
    """
    Set the configuration for noise testing.
    Args:
        weight_path (str): The path to the weight file.
        sensor (str): The sensor: sentinel or venus
        SNR (float): The signal to noise ratio.
        mtf_at_fe (float): The modulation transfer function at the Nyquist frequency.
        Seed (int) : The seed for the random number generator.
    Returns:
        Config: The modified configuration object.
    """
    IMG_SIZE = 2048 # Default
    
    if isinstance(weight_path, str) :
        cfgDir = Path(weight_path).parent
    else:
        cfgDir = weight_path.parent
    
    
    cfg = init_cfg(folder=f'{cfgDir}')
    band_sel = extract_first_band(weight_path.as_posix())
    assert band_sel is not None, f'Band not found in the weight path: {weight_path}'
    assert band_sel in [x for x in range(1,13,1)], f'Band not in the range [1,12]: {band_sel}'
    
    # Modify Config
    CHECKPOINT = weight_path
    cfg.load_from = f"{CHECKPOINT}"
    print(f'Loading from: {CHECKPOINT}')
    cfg.test_dataloader.dataset.pipeline = [{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': [band_sel]},
                        dict(type='LoadAnnotations', with_bbox=True),
                        dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                        dict(type='ImageCorruption', sensor=sensor, SNR=SNR, mtf_at_fe=mtf_at_fe), # 'gaussian', 
                        dict(
                            meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                    'height', 'width', 'instances', 'sample_idx', 
                                    'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                    'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                    'gt_bboxes_labels'),
                            type='PackDetInputs'),
                    ]

    work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
    cfg.work_dir = work_dir
    
    # Hook for using the SIoU intestead of the IoU:
    cfg.test_evaluator.type = 'SIoUCocoMetric'
    return cfg

### Test Functionality

In [2]:
TEST = False
if TEST:
    band_sel = 5 # BANDA SELEZIONATA

    weigths = list_weights(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band_sel}')
    assert len(weigths) > 0, 'No weights found'
    weight_path = weigths[0]
    cfg = set_NoiseTesting_config(weight_path, sensor='venus', SNR=5, mtf_at_fe = 0.2, Seed=18)
    # TEST
    runner = RUNNERS.build(cfg)
    work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
    runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
    output_test_data = runner.test()

# Cycle Loop

The maximum severity is 0.3. 

In [3]:
Results = {'Band':[], 'SNR':[], 'MTF':[], 'Precision':[], 'Recall':[], 'F1':[]}

SENSOR = 'venus'
BAND = 10 if SENSOR == 'venus' else 8
mtf_tgt = 0.2 if SENSOR == 'venus' else 0.3


SNR = 174 # Nominal SNR --> for S-2 at B8: 174 | for VENµS: 100. 
SNR_levels = np.geomspace(SNR, 1, 20)

for mtf_tgt in [0.2, 0.1, 0.05, 0.01, 0.001]:
    for SNR in SNR_levels:
        # TODO: Update weight path according to band_selected.
        weight_path = get_weight_b(BAND)
        cfg = set_NoiseTesting_config(weight_path, sensor=SENSOR, SNR=SNR, mtf_at_fe = mtf_tgt, Seed=18)

        # Run the test
        runner = RUNNERS.build(cfg)
        work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
        runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
        output_test_data = runner.test()
        
        ####### Save the results:
        P = output_test_data['coco/bbox_mAP']
        R = output_test_data['coco/bbox_AR@100']
        
        Results['Band'].append(BAND)
        Results['SNR'].append(SNR)
        Results['MTF'].append(mtf_tgt)
        
        Results['Precision'].append(P)
        Results['Recall'].append(R)
        try:
            F1 = 2 * (P * R) / (P + R)
        except ZeroDivisionError:
            F1 = 0
        Results['F1'].append(F1)
        

pd.DataFrame(Results).to_pickle(f'venus_SNR_study_b{BAND}.pkl')
print('Done')

Loading from: /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b10/18_BS_3_LR_0.0008_ME_30_OPT_SGD/epoch_30.pth
09/14 14:26:28 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 18
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (us

KeyError: 'SIoUCocoMetric is not in the mmdet::metric registry. Please check whether the value of `SIoUCocoMetric` is correct or it was registered as expected. More details can be found at https://mmengine.readthedocs.io/en/latest/advanced_tutorials/config.html#import-the-custom-module'

In [ ]:
Results = {'Band':[], 'SNR':[], 'MTF':[], 'Precision':[], 'Recall':[], 'F1':[]}

SENSOR = 'sentinel'
BAND = 10 if SENSOR == 'venus' else 8
mtf_tgt = 0.2 if SENSOR == 'venus' else 0.3


SNR = 174 # Nominal SNR --> for S-2 at B8: 174 | for VENµS: 100. 
SNR_levels = np.geomspace(SNR, 1, 20)

for SNR in SNR_levels:
    # TODO: Update weight path according to band_selected.
    weight_path = '/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Special/BS_4/LR_0.0005/IMG_2048/BANDS__b8/42_Optim_SGD/epoch_130.pth'
    cfg = set_NoiseTesting_config(weight_path, sensor=SENSOR, SNR=SNR, mtf_at_fe = mtf_tgt, Seed=18)

    # Run the test
    runner = RUNNERS.build(cfg)
    work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
    runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
    output_test_data = runner.test()
    
    ####### Save the results:
    P = output_test_data['coco/bbox_mAP']
    R = output_test_data['coco/bbox_AR@100']
    
    Results['Band'].append(BAND)
    Results['SNR'].append(SNR)
    Results['MTF'].append(mtf_tgt)
    
    Results['Precision'].append(P)
    Results['Recall'].append(R)
    try:
        F1 = 2 * (P * R) / (P + R)
    except ZeroDivisionError:
        F1 = 0
    Results['F1'].append(F1)

In [ ]:
import matplotlib.pyplot as plt

ax = pd.DataFrame(Results).plot(x='SNR', y=['F1', 'Precision', 'Recall'], title=f'Band {BAND} - MTF: {mtf_tgt}')
ax.invert_xaxis()
ax.set_ylabel('Metric Value')

plt.show()

In [6]:
pd.DataFrame(Results).to_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/TTA_OpticalModel.pkl')

# Plot

In [9]:
Spike = pd.read_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/TTA_SpikeNoise.pkl')
Guass = pd.read_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/TTA_GaussianNoise.pkl')

In [ ]:
import pandas as pd
from style import set_style
from matplotlib import pyplot as plt

# Set the style for the plots
set_style(scale_factor=.9, fontsize=9)

# Assuming Results is a dictionary or a similar data structure from which a DataFrame is created.
df = pd.DataFrame(Results)

def get_metrics_x_band(df, band):
    """
    Filter the DataFrame based on the specified band.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    band (int): The band value to filter the DataFrame by.

    Returns:
    pd.DataFrame: The filtered DataFrame containing only the data for the specified band.
    """
    return df[df['Band'] == band]

# Function to normalize the 'Severity' column
def normalize_column(column):
    """
    Normalize the values in a given pandas Series using min-max normalization.

    Parameters:
    column (pd.Series): The column to be normalized.

    Returns:
    pd.Series: The normalized column.
    """
    return (column - column.min()) / (column.max() - column.min())

# Plotting
fig = plt.figure(figsize=(12, 4))  # Adjust the figure size

# Loop through each band
for idx, band in enumerate(range(1, 13)):
    plt.subplot(2, 6, band)
    plt.title('($B_{'+str(band)+'}$)', x=0.1, y=0.85, fontdict={'fontsize': 9})
    
    # Retrieve and normalize data for Spike
    b_spike = get_metrics_x_band(Spike, band)
    x_spike = normalize_column(b_spike['Severity']).to_numpy()  # Normalized Severity for Spike
    y1_spike = b_spike['Precision'].to_numpy()
    y2_spike = b_spike['Recall'].to_numpy()
    y3_spike = b_spike['F1'].to_numpy()
    
    # Plot Spike data
    plt.plot(x_spike, y1_spike, label=f'$Precision$', linestyle='--', color='red')
    plt.plot(x_spike, y2_spike, label=f'$Recall$', linestyle='--', color='blue')
    plt.plot(x_spike, y3_spike, label=f'$F-1 Score$', linestyle='--', color='green')
    
    # Retrieve and normalize data for Guass
    b_guass = get_metrics_x_band(Guass, band)
    x_guass = normalize_column(b_guass['Severity']).to_numpy()  # Normalized Severity for Guass
    y1_guass = b_guass['Precision'].to_numpy()
    y2_guass = b_guass['Recall'].to_numpy()
    y3_guass = b_guass['F1'].to_numpy()
    
    # Plot Guass data
    plt.plot(x_guass, y1_guass, label=f'$Precision$', linestyle='-', color='red')
    plt.plot(x_guass, y2_guass, label=f'$Recall$', linestyle='-', color='blue')
    plt.plot(x_guass, y3_guass, label=f'$F-1 Score$', linestyle='-', color='green')
    
    # Set labels
    if idx in [0, 6]:
        plt.ylabel('Metric Value')
    if idx in [6, 7, 8, 9, 10, 11]:
        plt.xlabel('Severity [$\sigma$]')
    
    # Set limits and legend
    plt.ylim(0, 1)
    plt.xlim(0, 1)

plt.legend(loc='center', bbox_to_anchor=(-3.05, 2.4), ncol=6)

plt.tight_layout()
plt.savefig('/Data_large/marine/PythonProjects/MMDET/plots/SpikeVenus_normalized.png')
plt.show()
